In [9]:
!pip install langchain langchain-groq langchain-chroma langchain-huggingface sentence-transformers -q

In [10]:
import os
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Set Groq API key 
os.environ["GROQ_API_KEY"] = "API_KEY"

print("1. Loading Data...")
# Define private source text 
my_private_text = """
The secret project 'Phoenix' was launched in 2025 by an engineer named 'Ali Rad' in Tehran. 
The goal of this project is to convert plastic waste into jet fuel using carbon nanotubes. 
The initial budget for this project was 5 million dollars, funded by a Japanese investment firm.
"""
# Wrap the text into standard Document objects
docs = [Document(page_content=my_private_text)]

print("2. Creating Embeddings and Vector Database...")
# Initialize embedding model to convert text into vector representations
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a local vector store using Chroma DB
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)

# Initialize the retriever to fetch the top-1 most relevant chunk
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

print("3. Initializing LLM...")
# Initialize the LLM client via Groq
llm = ChatGroq(temperature=0, model_name="openai/gpt-oss-120b")

# Define the RAG prompt template enforcing source-only constraints
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a precise assistant. Answer the user's question strictly based on the provided 'Source Context'. If the answer cannot be found in the context, state 'I have no information about this.'\n\nSource Context:\n{context}"),
    ("human", "{question}")
])

print("\n--- Testing the RAG System ---\n")

# Define a targeted query that requires retrieval
question = "What is the Phoenix project and who built it?"
print(f"Question: {question}\n")

# --- Core RAG Execution Pipeline ---
# Step A: Retrieve relevant context from the vector database
relevant_docs = retriever.invoke(question)
context_text = relevant_docs[0].page_content 

# Step B: Generate the final response by combining context and prompt with the LLM
chain = prompt | llm
response = chain.invoke({"context": context_text, "question": question})

print("Answer (Generated by LLM):")
print(response.content)

1. Loading Data...
2. Creating Embeddings and Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3. Initializing LLM...

--- Testing the RAG System ---

Question: What is the Phoenix project and who built it?

Answer (Generated by LLM):
The Phoenix project is a secret initiative launched in 2025 that aims to convert plastic waste into jet fuel using carbon nanotubes. It was built by the engineer Ali Rad.
